# Food Pantry Optimizations

## Imports and Setup

In [63]:
import numpy as np
import pandas as pd

df = pd.read_csv('../data/processed/in_need_locations.csv')

## Minimize Average Distance

Use kmeans to cluster zipcodes, find centroid, new pantry location

In [40]:
from sklearn.cluster import KMeans
import folium

def optimize_kmeans(df: pd.DataFrame, n_clusters: int) -> np.ndarray:
    """
    Optimizes the selection of ZIP codes using KMeans clustering.
    """
    coords = df[['latitude', 'longitude']].values
    model = KMeans(n_clusters=n_clusters, random_state=42).fit(coords)
    df['cluster'] = model.labels_
    centroids = model.cluster_centers_

    # Choose closest real zip to each centroid
    selected_indices = []
    for center in centroids:
        distances = np.linalg.norm(coords - center, axis=1)
        selected_indices.append(distances.argmin())
    return np.array(selected_indices)

def make_map(df: pd.DataFrame, list_selected_indices: list[tuple[np.ndarray, str, str]]) -> folium.Map:
    """
    Creates a folium map with different layers for each optimization strategy.
    Each layer shows the selected ZIP codes for that strategy.

    Parameters:
    - df: DataFrame containing latitude, longitude, and zip code information.
    - list_selected_indices: List of tuples, each containing:
        - selected_indices: np.ndarray of selected indices for the strategy
        - color: str, color for the markers
        - label: str, label for the layer
    Returns:
    - folium.Map object with layers for each optimization strategy.
    """
    m = folium.Map(location=[df['latitude'].mean(), df['longitude'].mean()], zoom_start=8)

    # Layer for all ZIPs
    base = folium.FeatureGroup(name="All ZIPs", show=True)
    for _, row in df.iterrows():
        folium.CircleMarker(
            location=(row['latitude'], row['longitude']),
            radius=3,
            color='gray',
            fill=True,
            fill_opacity=0.4
        ).add_to(base)
    base.add_to(m)

    # Layers for each optimization strategy
    for indices, color, label in list_selected_indices:
        layer = folium.FeatureGroup(name=label, show=True)
        for idx in indices:
            row = df.iloc[idx]
            folium.Marker(
                location=(row['latitude'], row['longitude']),
                popup=f"{label}<br>ZIP: {row['zip']}",
                icon=folium.Icon(color=color, icon='cutlery', prefix='fa')
            ).add_to(layer)
        layer.add_to(m)

    folium.LayerControl().add_to(m)
    return m



selected_indices_1 = optimize_kmeans(df, n_clusters=5)
m = make_map(df, [(selected_indices_1, 'red', 'Minimize Average Distance')])
m


## Greedily Maximize Coverage Within R Radius Miles

In [64]:
from geopy.distance import geodesic

def optimize_greedy_coverage(D, optimizer, n_pantries, radius):
    covered = np.zeros(len(optimizer), dtype=bool)
    selected = []

    for _ in range(n_pantries):
        best_gain = 0
        best_idx = -1
        for i in range(len(optimizer)):
            if i in selected: 
                continue
            can_cover = (D[i] <= radius) & ~covered
            gain = optimizer[can_cover].sum()
            if gain > best_gain:
                best_gain, best_idx = gain, i
        selected.append(best_idx)
        covered |= (D[best_idx] <= radius)
    return np.array(selected)

def compute_distance_matrix(df: pd.DataFrame) -> np.ndarray:
    coords = list(zip(df['latitude'], df['longitude']))
    n = len(coords)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            D[i, j] = geodesic(coords[i], coords[j]).miles
    return D


### By Population

In [65]:
D = compute_distance_matrix(df)
populations = df['irs_estimated_population'].values

selected_indices_2 = optimize_greedy_coverage(D, populations, n_pantries=5, radius=10)
zip_codes = df.iloc[selected_indices_2]['zip'].values
display(df.iloc[selected_indices_2])
print("Greedy coverage by population zip codes:", zip_codes)
m = make_map(df, [(selected_indices_2, 'red', 'Greedy Population')])
m

,state_x,zip,poverty_rate,median_income_household,percent_lower_education,percent_higher_education,percent_snap_participation,per_capita_income,num_food_access,num_grocery,...,unacceptable_cities,state_y,county,timezone,area_codes,world_region,country,latitude,longitude,irs_estimated_population
50,MA,2150,0.214079,72122.0,0.616813,0.383187,0.257802,31526.0,18.0,0.0,...,NaN,MA,Suffolk County,America/New_York,"339, 617, 781, 857",NaN,US,42.39,-71.03,32310
8,MA,1840,0.325065,34630.0,0.695425,0.304575,0.584116,24230.0,5.0,0.0,...,NaN,MA,Essex County,America/New_York,"351, 978",NaN,US,42.71,-71.16,5500
26,MA,1151,0.175581,57113.0,0.568506,0.431494,0.389906,26701.0,2.0,1.0,...,Spfld,MA,Hampden County,America/New_York,413,NaN,US,42.15,-72.51,7610
1,MA,2325,0.707317,107024.5,0.200000,0.800000,0.091044,7596.0,0.0,0.0,...,Bridgewater State College,MA,Plymouth County,America/New_York,"508, 774",NaN,US,41.99,-70.96,10
53,MA,2745,0.119667,73154.0,0.535824,0.464176,0.220866,36353.0,3.0,1.0,...,NaN,MA,Bristol County,America/New_York,"508, 774",NaN,US,41.70,-70.95,21790


Greedy coverage by population zip codes: [2150 1840 1151 2325 2745]


### By Need Score

In [66]:


# Normalize features between 0 and 1
df['norm_poverty'] = df['poverty_rate'] / df['poverty_rate'].max()
df['norm_snap'] = df['percent_snap_participation'] / df['percent_snap_participation'].max()
df['norm_edu'] = df['percent_lower_education'] / df['percent_lower_education'].max()
df['norm_income'] = 1 - (df['per_capita_income'] / df['per_capita_income'].max())

# Composite score
df['need_score'] = (
    0.3 * df['norm_poverty'] +
    0.3 * df['norm_snap'] +
    0.1 * df['norm_edu'] +
    0.3 * df['norm_income']
)

need_scores = df['need_score'].values

selected_indices_3 = optimize_greedy_coverage(D, need_scores, n_pantries=5, radius=10)
zip_codes = df.iloc[selected_indices_3]['zip'].values
display(df.iloc[selected_indices_3])
print("Greedy coverage by need score zip codes:", zip_codes)
m = make_map(df, [(selected_indices_3, 'red', 'Greedy Need Score')])
m

,state_x,zip,poverty_rate,median_income_household,percent_lower_education,percent_higher_education,percent_snap_participation,per_capita_income,num_food_access,num_grocery,...,world_region,country,latitude,longitude,irs_estimated_population,norm_poverty,norm_snap,norm_edu,norm_income,need_score
26,MA,1151,0.175581,57113.0,0.568506,0.431494,0.389906,26701.0,2.0,1.0,...,NaN,US,42.15,-72.51,7610,0.248236,0.638882,0.733073,0.428403,0.467964
50,MA,2150,0.214079,72122.0,0.616813,0.383187,0.257802,31526.0,18.0,0.0,...,NaN,US,42.39,-71.03,32310,0.302663,0.422423,0.795365,0.325113,0.394596
8,MA,1840,0.325065,34630.0,0.695425,0.304575,0.584116,24230.0,5.0,0.0,...,NaN,US,42.71,-71.16,5500,0.459575,0.957106,0.896732,0.481301,0.659068
12,MA,1610,0.334189,43757.0,0.588830,0.411170,0.424560,20889.0,34.0,1.0,...,NaN,US,42.25,-71.81,16900,0.472475,0.695664,0.759281,0.552823,0.592217
53,MA,2745,0.119667,73154.0,0.535824,0.464176,0.220866,36353.0,3.0,1.0,...,NaN,US,41.70,-70.95,21790,0.169185,0.361901,0.690931,0.221780,0.294953


Greedy coverage by need score zip codes: [1151 2150 1840 1610 2745]


## All Optimal Locations Together

In [67]:
m = make_map(df, [
        (selected_indices_1, 'red', 'Minimize Average Distance'), 
        (selected_indices_2, 'blue', 'Greedy Population'), 
        (selected_indices_3, 'green', 'Greedy Need Score'),
        ]
    )
m